In [1]:
# =============================================================================
# CELL 1 — IMPORTS
# =============================================================================

import yfinance as yf
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from scipy.optimize import minimize

In [2]:
# =============================================================================
# CELL 2 — DATA
# =============================================================================

def fetch_intraday(ticker, interval="15m", total_days=59):
    end   = datetime.datetime.today()
    start = end - datetime.timedelta(days=total_days)
    df    = yf.download(
        ticker,
        start=start.strftime("%Y-%m-%d"),
        end=end.strftime("%Y-%m-%d"),
        interval=interval,
        auto_adjust=True,
        progress=False
    )
    return df[["Close"]].rename(columns={"Close": ticker})

meta         = fetch_intraday("META")
qqq          = fetch_intraday("QQQ")
data         = pd.concat([meta, qqq], axis=1).dropna()
data.columns = ["META", "QQQ"]

meta_vals = data["META"].values.astype(float)
qqq_vals  = data["QQQ"].values.astype(float)

print(f"Data shape: {data.shape}")
print(f"Window: {data.index[0].date()} → {data.index[-1].date()}")

Data shape: (1092, 2)
Window: 2026-02-25 → 2026-04-24


In [3]:
# =============================================================================
# CELL 3 — KALMAN FILTER (STAGE 1 PARAMETERS)
# =============================================================================

def kalman_filter(meta, qqq, delta=1e-4, Ve=0.001):
    n  = len(meta)
    Vw = delta / (1 - delta) * np.eye(2)

    beta = np.zeros((n, 2))
    P    = np.zeros((n, 2, 2))
    e    = np.zeros(n)
    Q    = np.zeros(n)

    beta[0] = [0, 1]
    P[0]    = np.eye(2)

    for t in range(1, n):
        F         = np.array([1.0, qqq[t]])
        beta_pred = beta[t-1]
        P_pred    = P[t-1] + Vw

        e[t] = meta[t] - F @ beta_pred
        Q[t] = F @ P_pred @ F + Ve

        K       = (P_pred @ F) / Q[t]
        beta[t] = beta_pred + K * e[t]
        P[t]    = P_pred - np.outer(K, F) @ P_pred

    return beta, e, Q

beta, e, Q = kalman_filter(meta_vals, qqq_vals)

print(f"β₁ mean: {beta[:,1].mean():.4f}")
print(f"β₁ std:  {beta[:,1].std():.4f}")
print(f"β₁ range: [{beta[:,1].min():.4f}, {beta[:,1].max():.4f}]")

β₁ mean: 1.0328
β₁ std:  0.0382
β₁ range: [0.9245, 1.1008]
